# JHBI DS App Pricing — Piecewise Linear Model

**3 inflection points:** `$250M` · `$1B` · `$5B` in FI total assets  
**5 apps:** Zelle Memo Intelligence, Churn Sentinel, CommercialSignal, Gen. Wealth Deflection, Anomaly Detection  
**CU discount:** 12% cooperative pricing accommodation

---

### How it works

Instead of 4 discrete step-function tiers (which create cliff edges at `$250M`, `$1B`, `$5B`),  
the piecewise linear model interpolates **continuously** between anchor prices at each breakpoint.

```
Price
  │                                      ●────────── (cap)
  │                              ●──────╱
  │                      ●──────╱
  │              ●───────╱
  │  (flat)──────╱
  └──────────────┼───────┼───────┼──────────── Assets
               $250M   $1B    $5B
```

Within each of the 4 segments, price changes linearly with asset size.  
This eliminates arbitrary pricing cliffs and is easier to defend to procurement teams.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 1 · Model Definition

In [ ]:
# ── Breakpoints (FI total assets in $M) ────────────────────────────
# Segment 1:  $0       → $250M   (tiny community banks & CUs)
# Segment 2:  $250M    → $1B     (core community market)
# Segment 3:  $1B      → $5B     (larger regional FIs)
# Segment 4:  $5B+              (cap — held flat)

BREAKPOINTS = [0, 250, 1_000, 5_000, 50_000]  # $M, last point holds the cap

# ── Annual prices at each breakpoint (bank pricing, USD/year) ──────
# Each row: [at $0M, at $250M, at $1B, at $5B, at $50B+]
# Source: JHBI DS Revenue Update, slide 22
APPS = {
    'Zelle Memo Intelligence':  [4_000,  5_000,  8_000, 15_000, 25_000],
    'Churn Sentinel':           [6_500,  8_000, 12_000, 22_000, 35_000],
    'CommercialSignal':         [5_000,  6_000, 10_000, 18_000, 28_000],
    'Gen. Wealth Deflection':   [6_500,  8_000, 12_000, 20_000, 30_000],
    'Anomaly Detection':        [7_000,  9_000, 14_000, 24_000, 40_000],
}

CU_DISCOUNT = 0.12   # 12% cooperative discount for credit unions

COLORS = {
    'Zelle Memo Intelligence':  '#06B6D4',
    'Churn Sentinel':           '#059669',
    'CommercialSignal':         '#1D4ED8',
    'Gen. Wealth Deflection':   '#7C3AED',
    'Anomaly Detection':        '#DC2626',
}

# ── Core pricing function ───────────────────────────────────────────
def pwl_price(assets_M: float, app: str, is_cu: bool = False) -> float:
    """
    Piecewise linear annual subscription price for a given FI.

    Parameters
    ----------
    assets_M : float  — FI total assets in $millions
    app      : str    — app name key from APPS dict
    is_cu    : bool   — True to apply 12% CU cooperative discount

    Returns
    -------
    float — annual price in USD (rounded to nearest dollar)
    """
    prices = APPS[app]
    # np.interp clamps at boundaries (flat extrapolation outside range)
    price = float(np.interp(assets_M, BREAKPOINTS, prices))
    if is_cu:
        price *= (1 - CU_DISCOUNT)
    return round(price)


def price_all_apps(assets_M: float, is_cu: bool = False) -> dict:
    """Return {app: annual_price} for all 5 apps at a given asset size."""
    return {app: pwl_price(assets_M, app, is_cu) for app in APPS}


def bundle_price(assets_M: float, apps: list, is_cu: bool = False,
                 bundle_discount: float = 0.20) -> dict:
    """
    Calculate bundle pricing with a discount off the sum of individual prices.

    Parameters
    ----------
    apps            : list of app names to bundle
    bundle_discount : fractional discount (default 0.20 = 20%)
    """
    individual = {app: pwl_price(assets_M, app, is_cu) for app in apps}
    total_list  = sum(individual.values())
    bundle_total = round(total_list * (1 - bundle_discount))
    return {
        'individual_prices': individual,
        'list_total':        total_list,
        'bundle_discount':   f'{bundle_discount:.0%}',
        'bundle_price':      bundle_total,
        'savings':           total_list - bundle_total,
    }

print('Model defined. Functions: pwl_price(), price_all_apps(), bundle_price()')

## 2 · Price Table — Spot Check

In [ ]:
test_sizes = [75, 150, 250, 500, 750, 1_000, 2_000, 3_500, 5_000, 10_000, 25_000]

rows = []
for sz in test_sizes:
    label = f'${sz:,}M' if sz < 1_000 else f'${sz//1_000:,}B'
    row = {'Assets': label}
    for app in APPS:
        short = app.split()[0]  # first word as column name
        row[short + ' (Bank)'] = f"${pwl_price(sz, app):,}"
        row[short + ' (CU)']   = f"${pwl_price(sz, app, is_cu=True):,}"
    rows.append(row)

df = pd.DataFrame(rows).set_index('Assets')

# Show bank prices only for readability
bank_cols = [c for c in df.columns if '(Bank)' in c]
bank_df = df[bank_cols].copy()
bank_df.columns = [c.replace(' (Bank)', '') for c in bank_df.columns]
print('Annual Bank Pricing by Asset Size:\n')
print(bank_df.to_string())

## 3 · Pricing Curves

In [ ]:
def format_asset(v, p):
    if v >= 1_000: return f'${v/1_000:.0f}B'
    return f'${v:.0f}M'

def format_price(v, p):
    if v >= 1_000: return f'${v/1_000:.0f}K'
    return f'${v:.0f}'

assets_range = np.linspace(10, 50_000, 3000)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
fig.patch.set_facecolor('#F0F4FA')
fig.suptitle('JHBI DS App Pricing — Piecewise Linear Model\n'
             'Inflection points: $250M  ·  $1B  ·  $5B',
             fontsize=14, fontweight='bold', color='#1E2761', y=1.02)

for ax, is_cu in zip(axes, [False, True]):
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#CBD5E1')
    ax.tick_params(colors='#374151', labelsize=9)

    for app, color in COLORS.items():
        prices = [pwl_price(a, app, is_cu) for a in assets_range]
        ax.plot(assets_range, prices, color=color, linewidth=2.3,
                label=app, zorder=3)
        # Inflection point dots
        for bp in [250, 1_000, 5_000]:
            p = pwl_price(bp, app, is_cu)
            ax.scatter(bp, p, color=color, s=45, zorder=5, linewidths=0)

    # Inflection point guides
    for bp, lbl in [(250, '$250M'), (1_000, '$1B'), (5_000, '$5B')]:
        ax.axvline(bp, color='#94A3B8', linewidth=0.9, linestyle='--', alpha=0.7, zorder=1)
        ax.text(bp * 1.04, 41_500, lbl, fontsize=8, color='#64748B', va='top')

    ax.set_xscale('log')
    ax.set_xlim(30, 52_000)
    ax.set_ylim(0, 44_000)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(format_asset))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_price))
    ax.set_xlabel('FI Total Assets', fontsize=10, color='#374151', labelpad=8)
    ax.set_ylabel('Annual Subscription Price (USD)', fontsize=10, color='#374151')
    suffix = ' — Credit Union (−12%)' if is_cu else ' — Bank'
    ax.set_title(f'Pricing Curves{suffix}', fontsize=11, fontweight='bold',
                 color='#1E2761', pad=10)
    ax.grid(axis='y', color='#E2E8F0', linewidth=0.6, zorder=0)

axes[0].legend(fontsize=8.5, framealpha=0.95, loc='upper left',
               edgecolor='#CBD5E1', facecolor='white')

plt.tight_layout()
plt.savefig('JHBI_Pricing_Curves.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved: JHBI_Pricing_Curves.png')

## 4 · Slope Analysis — Marginal Price per $1B of Assets

The slope of each segment tells you how aggressively pricing scales within that tier range.

In [ ]:
segments = [
    ('$0 → $250M',   0,     250),
    ('$250M → $1B',  250,   1_000),
    ('$1B → $5B',    1_000, 5_000),
]

print(f"{'App':<30} {'Seg 1 ($/1B assets)':>22} {'Seg 2 ($/1B assets)':>22} {'Seg 3 ($/1B assets)':>22}")
print('-' * 100)
for app in APPS:
    slopes = []
    for _, lo, hi in segments:
        rise = pwl_price(hi, app) - pwl_price(lo, app)   # $ increase
        run  = (hi - lo) / 1_000                          # convert $M → $B
        slopes.append(f"${rise/run:,.0f}")
    print(f"{app:<30} {slopes[0]:>22} {slopes[1]:>22} {slopes[2]:>22}")

## 5 · Pricing Quote Tool

Enter any FI's asset size to get an instant quote for all apps.

In [ ]:
def quote(assets_M: float, is_cu: bool = False, apps_selected: list = None,
          bundle_discount: float = 0.0):
    """
    Generate a clean pricing quote for a given FI.

    Parameters
    ----------
    assets_M        : FI total assets in $M
    is_cu           : True for credit union (12% discount applied)
    apps_selected   : list of app names to include (None = all 5)
    bundle_discount : additional bundle discount fraction (e.g. 0.20 = 20%)
    """
    if apps_selected is None:
        apps_selected = list(APPS.keys())

    fi_type = 'Credit Union' if is_cu else 'Bank'
    label   = f'${assets_M:,}M' if assets_M < 1_000 else f'${assets_M/1_000:,.1f}B'

    print(f"\n{'═'*55}")
    print(f"  JHBI DS Pricing Quote")
    print(f"  FI Assets: {label}  |  Type: {fi_type}")
    print(f"{'═'*55}")

    total = 0
    for app in apps_selected:
        p = pwl_price(assets_M, app, is_cu)
        total += p
        print(f"  {app:<32}  ${p:>8,} /yr")

    print(f"{'─'*55}")
    print(f"  {'List Total':<32}  ${total:>8,} /yr")

    if bundle_discount > 0:
        discounted = round(total * (1 - bundle_discount))
        savings    = total - discounted
        print(f"  Bundle Discount ({bundle_discount:.0%}){'':14}  -${savings:>7,}")
        print(f"  {'Bundle Price':<32}  ${discounted:>8,} /yr")

    print(f"{'═'*55}")
    print(f"  Monthly equivalent:   ${total/12:>8,.0f} /mo")
    print(f"{'═'*55}\n")


# ── Example quotes ────────────────────────────────────────────
# Small community bank, $400M assets
quote(400)

# Mid-size credit union, $1.2B assets
quote(1_200, is_cu=True)

# Regional bank, $3.5B assets, all 5 apps, 20% bundle discount
quote(3_500, bundle_discount=0.20)

## 6 · ARR Sensitivity — Penetration × Asset Distribution

Models projected ARR given penetration rates across JH's 1,660 core FIs,  
segmented by asset size tier.

In [ ]:
# Approximate distribution of JH's 1,660 core FIs by asset tier
# (44% CU / 56% bank mix per JHBI revenue update)
FI_DISTRIBUTION = [
    # (tier_label, midpoint_assets_M, n_banks, n_cus)
    ('< $250M',      125,  520, 380),
    ('$250M–$1B',    500,  280, 200),
    ('$1B–$5B',    2_000,  100,  80),
    ('$5B+',      10_000,   30,  70),
]

def arr_model(app: str, penetration_by_tier: list,
              churn_rate_annual: float = 0.08) -> dict:
    """
    Calculate projected ARR for a single app.

    Parameters
    ----------
    app                  : app name string
    penetration_by_tier  : list of 4 penetration rates [tier1, tier2, tier3, tier4]
    churn_rate_annual    : annual subscription churn rate (default 8%)
    """
    total_arr = 0
    rows = []
    for (tier_lbl, mid_M, n_banks, n_cus), pen in zip(FI_DISTRIBUTION, penetration_by_tier):
        bank_price = pwl_price(mid_M, app, is_cu=False)
        cu_price   = pwl_price(mid_M, app, is_cu=True)
        subs_banks = round(n_banks * pen)
        subs_cus   = round(n_cus   * pen)
        tier_arr   = subs_banks * bank_price + subs_cus * cu_price
        total_arr += tier_arr
        rows.append({'Tier': tier_lbl, 'Banks': subs_banks, 'CUs': subs_cus,
                     'Avg Price (Bank)': f'${bank_price:,}',
                     'Avg Price (CU)':   f'${cu_price:,}',
                     'Tier ARR':         f'${tier_arr:,.0f}'})

    net_arr = round(total_arr * (1 - churn_rate_annual))
    return {'app': app, 'gross_arr': total_arr, 'net_arr': net_arr,
            'churn_rate': churn_rate_annual, 'detail': pd.DataFrame(rows)}


# ── Run model for Churn Sentinel at 6% penetration across tiers
result = arr_model('Churn Sentinel', penetration_by_tier=[0.04, 0.07, 0.10, 0.12])
print(f"ARR Model: {result['app']}")
print(result['detail'].to_string(index=False))
print(f"\nGross ARR:  ${result['gross_arr']:,.0f}")
print(f"Net ARR (−{result['churn_rate']:.0%} churn):  ${result['net_arr']:,.0f}")

In [ ]:
# ── Full platform ARR across all 5 apps ─────────────────────────
# Penetration assumptions per app (conservative, matching revenue update model)
APP_PENETRATION = {
    'Zelle Memo Intelligence':  [0.02, 0.03, 0.05, 0.08],   # Zelle-only FIs
    'Churn Sentinel':           [0.04, 0.07, 0.10, 0.12],
    'CommercialSignal':         [0.03, 0.05, 0.08, 0.10],
    'Gen. Wealth Deflection':   [0.02, 0.04, 0.06, 0.09],
    'Anomaly Detection':        [0.02, 0.03, 0.05, 0.08],
}

print(f"{'App':<32} {'Gross ARR':>14} {'Net ARR (−8%)':>16}")
print('─' * 65)
grand_gross = grand_net = 0
for app, pen in APP_PENETRATION.items():
    r = arr_model(app, pen)
    grand_gross += r['gross_arr']
    grand_net   += r['net_arr']
    print(f"{app:<32} ${r['gross_arr']:>12,.0f}  ${r['net_arr']:>12,.0f}")

print('─' * 65)
print(f"{'TOTAL PLATFORM ARR':<32} ${grand_gross:>12,.0f}  ${grand_net:>12,.0f}")

## 7 · Adjust the Model

To change pricing, edit `APPS` in Cell 2.  
To change inflection points, edit `BREAKPOINTS` — the function automatically re-interpolates.

```python
# Example: shift the middle breakpoint from $1B to $750M
BREAKPOINTS = [0, 250, 750, 5_000, 50_000]

# Example: add a 4th inflection point at $15B
BREAKPOINTS = [0, 250, 1_000, 5_000, 15_000, 50_000]
# (update APPS price lists to have 6 values instead of 5)
```

Quick lookup for any FI:

```python
pwl_price(850, 'Churn Sentinel')              # $850M bank
pwl_price(850, 'Churn Sentinel', is_cu=True)  # $850M credit union
price_all_apps(2_000)                         # all 5 apps for $2B bank
quote(1_500, is_cu=True, bundle_discount=0.20) # quote with bundle
```